# Unit 04: Parallel computing and acceleration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weizj99/computational-economics-mp/blob/main/notebooks/04_parallel.ipynb)

This notebook runs the CPU and GPU code of [unit 04](https://computational-economics.zhongjiwei.org/modules/04_parallel/). The CPU cells run on any machine; the GPU cells need an NVIDIA GPU. In Colab, choose one under Runtime > Change runtime type (T4 GPU) before you start, or run the CPU cells first and switch later.

Run the cells from top to bottom. The first cell fetches the course code when the notebook runs in Google Colab; on your own computer it uses the repository the notebook is in.

In [ ]:
# Set up. In Colab this fetches the course code; on your own computer it uses
# the copy of the repository that this notebook is in.
import os
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/weizj99/computational-economics-mp"
if "google.colab" in sys.modules:
    folder = Path("/content") / REPOSITORY.rsplit("/", 1)[-1]
    if not (folder / "run.py").exists():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", "main", REPOSITORY, str(folder)],
                       check=True)
    os.chdir(folder)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-teaching.txt"], check=True)
else:
    folder = Path.cwd().resolve()
    while not (folder / "run.py").exists() and folder != folder.parent:
        folder = folder.parent
    if not (folder / "run.py").exists():
        raise SystemExit("Open this notebook in Colab, or from inside the course repository.")
    os.chdir(folder)


def run(*command):
    """Run a course script with this notebook's Python and show its output as it comes."""
    process = subprocess.Popen([sys.executable, *command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"{command[0]} stopped with exit code {process.returncode}")


print("Course code in", Path.cwd())

## Time the CPU workers

The first 16 of unit 03's timed economies, solved with 1, 2, 4, … worker processes up to the number of cores. Every line should end with “matches saved results: True”. On four cores it takes about four minutes.

In [ ]:
run("lessons/04_parallel/scaling.py", "--cases", "16")

## Lesson 02: one call or one call per market

The market shares of 2,000 markets, solved in one call and in one call per market. It takes about 30 seconds.

In [ ]:
run("lessons/04_parallel/loop_vs_vectorized.py")

## Extra exercise: the output equations

Newton's method and bisection on the output equations of unit 02's baseline economy. It takes about ten seconds.

In [ ]:
run("lessons/04_parallel/output_root.py")

## On a GPU

The next cell looks for a GPU. If it finds one, it solves one economy on the CPU and on the GPU and compares the two; `"passed": true` means they agree. The first GPU solve compiles the kernels, so it takes longer than later ones.

In [ ]:
# Look for a GPU, then solve one economy on the CPU and on the GPU and compare.
# The search runs in a separate process, so a missing or broken PyTorch only prints a message.
probe = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0) "
                        "if torch.cuda.is_available() else '')"], capture_output=True, text=True)
gpu = probe.stdout.strip() if probe.returncode == 0 else None
if probe.returncode != 0:
    print("PyTorch could not be loaded, so the GPU cells are skipped.\n"
          "Install a CUDA build of PyTorch to run them, or use Colab.")
elif not gpu:
    print("No GPU found, so the GPU cells are skipped. The CPU cells above do not need one.\n"
          "In Colab: Runtime > Change runtime type > T4 GPU, then run the first cell and this one again.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "threadpoolctl"], check=True)
    print("GPU:", gpu)
    run("lessons/04_parallel/gpu_economy.py", "--case", "candidate_000", "--compare")

The last cell runs all three GPU checks against the CPU solver: 17 economies on the single-economy GPU code, 8 on the batched engine and the market-share block. Its last line should read “All GPU checks passed.” It takes several minutes.

In [ ]:
# All three GPU checks, if the cell above found a GPU.
if gpu:
    run("lessons/04_parallel/check_gpu.py")
else:
    print("No GPU, so the GPU checks are skipped.")